# DQVP operationalization pipeline: from requirement to validated data

This notebook runs the example end to end — from a data-quality requirement through to validated data — across four validation engines and three data formats.

```
DQR catalogue (requirements/*.json)
   ──[dqr_to_dqvp.py: pattern → DQVP]───────► DQVP policy (odrl:constraint on a Permission)
   ──[EYE rules/plan.n3 + config + vocab]──► Policy Checker (workflow of OP: input→output)
   ──[graph templates + 1 Jinja binder]────► check code (GX, dbt, SodaCL, PyDeequ)
                                          └─► load code per (engine, format): csv / json / parquet
   ──[execute on the data]──────────────────► verdicts + dqv:QualityMeasurement
```

- Requirement: each DQR JSON instantiates a pattern (`DQRP*`); the mapper fills its parameters into a `dqvp:DQConstraint`.
- Contract: the provider offers the asset for use under quality conditions (`odrl:constraint` on an `odrl:Permission`; no "validate" action).
- Plan: typed operations with an explicit `input → output` shape.
- Code: the check code is format-agnostic; only the Load operation depends on the data format, so the same plan validates CSV, JSON or Parquet.

## 0. The requirements

The paper's running example places three data-quality requirements (DQRs) on a *patient-summary* asset — one per ISO/IEC 25012 dimension, each hitting a different ODRL modelling barrier:

| # | Requirement (plain English) | Dimension | Barrier(s) |
|---|---|---|---|
| **DQR1** | `nationalHealthcarePatientID` must be non-null in **at least 99%** of records | Completeness | B1 |
| **DQR2** | `countryOfAffiliation` must conform to **ISO 3166** in **at least 80%** of records | Compliance | B1 + B2 |
| **DQR3** | where `gender = "male"`, `pregnancyHistory` must be **completely null** | Consistency | B1 + B3 |

We follow these end to end: **plain English → structured requirement → DQVP policy → validation code → verdict**. First, each requirement is captured as a structured DQR — a catalogue entry that instantiates a reusable pattern (`DQRP*`) and fills its parameters.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
PIPE = "operationalization-pipeline"
sys.path.insert(0, PIPE)
from rdflib import RDF, Graph, Namespace
import codegen as C
import dqr_to_dqvp as D

EX = Namespace("https://example.org/ehds/")

# the three requirements as structured DQRs: natural-language statement + pattern + parameters
reqs = D.load_requirements()
for r in reqs:
    print(f"{r['id']}  pattern={r['pattern']['id']}  dimension={r['qualityDimension']['dimension']}")
    print(f'  statement : "{r["statement"]}"')
    print(f"  parameters: {json.dumps(r['parameters'])}\n")

# one full catalogue entry, for reference
print("full DQR catalogue entry —", reqs[0]["id"] + ":")
print(json.dumps(reqs[0], indent=2))

DQR1EH  pattern=DQRP2  dimension=Completeness
  statement : "In the PatientIdentification entity, the nationalHealthcarePatientID attribute must contain at least 99 percent non-empty values"
  parameters: {"entityName": "PatientIdentification", "datasetName": "PatientSummary", "attributeName": "nationalHealthcarePatientID", "operator": "at least", "percentageValue": "99"}

DQR2EH  pattern=DQRP3  dimension=Compliance
  statement : "In the PatientIdentification entity, the countryOfAffiliation attribute must follow the ISO 3166 standard in at least 80 percent of records"
  parameters: {"entityName": "PatientIdentification", "datasetName": "PatientSummary", "attributeName": "PatientSummary.PatientSummaryHeader.PatientIdentification.countryOfAffiliation", "standardName": "ISO 3166", "percentageValue": "80"}

DQR3EH  pattern=DQRP4  dimension=Consistency
  statement : "In the PatientIdentification entity, when the gender attribute has the values {male}, the pregnancyHistory attribute of Pati

### From requirement to DQVP policy

Mapping is a per-*pattern* lookup, not per-requirement code: each pattern fixes which DQVP metric (`leftOperand`) and which constructs its parameters fill. The result is one `dqvp:DQConstraint` per requirement, grouped conjunctively under a single `odrl:Permission`.

| Pattern (DQR) | Metric `leftOperand` | DQVP constructs used |
|---|---|---|
| DQRP2 — DQR1 | `dqvp:NonNullRate` | `targetAttribute` (B1) |
| DQRP3 — DQR2 | `dqvp:ReferenceConformance` | `targetAttribute` (B1), `referenceStandard` (B2) |
| DQRP4 — DQR3 | `dqvp:NullRate` | `targetAttribute` (B1), `filter`/`RowFilter` (B3) |

In [2]:
# map each structured requirement to its DQVP constraint (pattern -> constraint shape),
# then assemble the policy that the rest of the pipeline consumes
for r in reqs:
    name, block = D.constraint(r)
    print(f"{r['id']}  ->  ex:{name}")
    print("   " + block.replace("\n", "\n   "), "\n")

POLICY = Path(PIPE, "artifacts"); POLICY.mkdir(exist_ok=True)
POLICY = POLICY / "policy_from_dqr.ttl"
POLICY.write_text(D.build_policy(reqs))
print("assembled DQVP policy ->", POLICY, "\n")
print(POLICY.read_text())

DQR1EH  ->  ex:c-completeness
   ex:c-completeness a dqvp:DQConstraint ; rdfs:label "c-completeness" ;
       odrl:leftOperand     dqvp:NonNullRate ;
       dqvp:targetAttribute ab:nationalHealthcarePatientID ;
       odrl:operator        odrl:gteq ;
       odrl:rightOperand    "99"^^xsd:decimal . 

DQR2EH  ->  ex:c-compliance
   ex:c-compliance a dqvp:DQConstraint ; rdfs:label "c-compliance" ;
       odrl:leftOperand     dqvp:ReferenceConformance ;
       dqvp:targetAttribute ab:countryOfAffiliation ;
       dqvp:referenceStandard <https://www.iso.org/iso-3166-country-codes.html> ;
       odrl:operator        odrl:gteq ;
       odrl:rightOperand    "80"^^xsd:decimal . 

DQR3EH  ->  ex:c-consistency
   ex:c-consistency a dqvp:DQConstraint ; rdfs:label "c-consistency" ;
       odrl:leftOperand     dqvp:NullRate ;
       dqvp:targetAttribute ab:pregnancyHistory ;
       dqvp:filter [ a dqvp:RowFilter ;
                     dqvp:filterAttribute ab:gender ;
                     dqvp:filter

## 1. Compile to the Policy Checker

One `pc:PolicyChecker` `pc:accordingTo` the policy: a shared `Load` (Dataset input, Table output) feeds parallel per-check branches (`Filter → Measure → Compare → Report`, plus `Resolve → CodeList` for the reference check). Every operation has the `input → output` shape; parameters are added on top.

In [3]:
plan = C.build_plan("dataset_csv.ttl", policy=POLICY)
plan.serialize(destination=f"{PIPE}/artifacts/plan.ttl", format="turtle")
PC = C.PC

chk = next(plan.subjects(RDF.type, PC.PolicyChecker))
wf  = plan.value(chk, PC.hasWorkflow)

def branch(op):                                   # which check this op belongs to ("·shared" Load)
    c = plan.value(op, PC.forCheck)
    return C.local(c) if c else "·shared"
def opkind(op):                                   # Load / Filter / Measure / …
    return C.local(next(x for x in plan.objects(op, RDF.type) if x != PC.Operation))

print(f"pc:PolicyChecker {C.local(chk)}  accordingTo {C.local(plan.value(chk, PC.accordingTo))}\n")
print(f"  {'branch':16s} {'op':8s} input -> output")
for op in sorted(plan.objects(wf, PC.hasOp), key=lambda o: (branch(o), int(plan.value(o, PC.order)))):
    ins = [C.local(x) for x in plan.objects(op, PC.input)]
    out = [C.local(x) for x in plan.objects(op, PC.output)]
    ps  = {str(plan.value(p, PC.name)): str(plan.value(p, PC.value)) for p in plan.objects(op, PC.parameter)}
    print(f"  {branch(op):16s} {opkind(op):8s} {ins} -> {out}" + (f"   {ps}" if ps else ""))

pc:PolicyChecker checker  accordingTo ps-dq-offer-01

  branch           op       input -> output
  c-completeness   Filter   ['table'] -> ['ftable']
  c-completeness   Measure  ['ftable'] -> ['mval']   {'mostly': '0.99', 'column': 'nationalHealthcarePatientID', 'complement': '1'}
  c-completeness   Compare  ['mval'] -> ['verdict']   {'sqlOp': '>=', 'pyOp': '>=', 'threshold': '99'}
  c-completeness   Report   ['verdict'] -> ['reportdoc']
  c-compliance     Resolve  [] -> ['codes']   {'referenceStandard': 'https://www.iso.org/iso-3166-country-codes.html'}
  c-compliance     Filter   ['table'] -> ['ftable']
  c-compliance     Measure  ['codes', 'ftable'] -> ['mval']   {'complement': '20', 'column': 'countryOfAffiliation', 'mostly': '0.8', 'needsCodes': 'true'}
  c-compliance     Compare  ['mval'] -> ['verdict']   {'pyOp': '>=', 'threshold': '80', 'sqlOp': '>='}
  c-compliance     Report   ['verdict'] -> ['reportdoc']
  c-consistency    Filter   ['table'] -> ['ftable']   {'filterColumn': 

### The Load operation's `Dataset` input carries the data format

In [4]:
src = plan.value(next(plan.subjects(RDF.type, PC.Load)), PC.input)
g = Graph(); g.bind("pc", PC)
for tr in plan.triples((src, None, None)): g.add(tr)
print(g.serialize(format="turtle"))

@prefix pc: <https://feed-upc.github.io/DQVP-position/pc#> .

<https://example.org/ehds/ps-dq-offer-01/checker/source> a pc:Dataset ;
    pc:format "csv" ;
    pc:locator "../data/Patient_Summary.csv" ;
    pc:ofAsset <https://w3id.org/ehds/patsum/asset/PatientSummary> .




## 2. Generate the check code

Four engines from the same workflow. This code is format-agnostic: it runs over the loaded table.

In [5]:
def show(s, n=96):
    return "\n".join((l[:n] + " …]") if len(l) > n else l for l in s.splitlines())

code_by_engine = C.generate(plan)
for cid in ["c-completeness", "c-compliance", "c-consistency"]:
    print(f"=== {cid} ===")
    for e in sorted(code_by_engine):
        print(f"  [{e:6s}] {show(code_by_engine[e][cid])}")
    print()

=== c-completeness ===
  [dbt   ] -- c-completeness
with m as (select 100.0*count("nationalHealthcarePatientID")/nullif(count(*),0) as r from {{ sou …]
select r from m where not (r >= 99)
  [deequ ]     .hasCompleteness("nationalHealthcarePatientID", lambda x: x >= 0.99)  # c-completeness
  [gx    ] validator.expect_column_values_to_not_be_null(column="nationalHealthcarePatientID", mostly=0.99)
  [sodacl]   - missing_percent(nationalHealthcarePatientID) <= 1:
      name: c-completeness

=== c-compliance ===
  [dbt   ] -- c-compliance
with m as (select 100.0*count(*) filter (where "countryOfAffiliation" in (select code from codes …]
select r from m where not (r >= 80)
  [deequ ]     .isContainedIn("countryOfAffiliation", CODE_LIST)  # c-compliance
  [gx    ] validator.expect_column_values_to_be_in_set(column="countryOfAffiliation", value_set=CODE_LIST,  …]
  [sodacl]   - invalid_percent(countryOfAffiliation) <= 20:
      name: c-compliance
      valid values: ['AD', 'AE', 'AF', 'AG', 'A

## 3. The data-format dimension

Only the Load operation depends on the format. The descriptor (`dataset_*.ttl`) sets `pc:format` and `pc:locator` on the asset; the binder renders the matching load template.

In [6]:
plan_csv     = C.build_plan("dataset_csv.ttl", policy=POLICY)
plan_json    = C.build_plan("dataset_json.ttl", policy=POLICY)
plan_parquet = C.build_plan("dataset_parquet.ttl", policy=POLICY)

for p in (plan_csv, plan_json, plan_parquet):
    fmt, _ = C.dataset_of(p)
    print(f"format = {fmt}")
    for eng in ["gx", "duckdb", "dbt"]:
        print(f"   load[{eng:6s}] {C.load_code(p, eng)}")
    print()

print("the CHECK code is identical across formats — e.g. GX c-completeness:")
print("  ", C.generate(plan_csv)["gx"]["c-completeness"])

format = csv
   load[gx    ] pd.read_csv('/home/acraf/psr/Fdatavalidation-1/PositionPaper/data/Patient_Summary.csv')
   load[duckdb] read_csv_auto('/home/acraf/psr/Fdatavalidation-1/PositionPaper/data/Patient_Summary.csv')
   load[dbt   ] # dbt source (dbt-duckdb): external_location: read_csv_auto('/home/acraf/psr/Fdatavalidation-1/PositionPaper/data/Patient_Summary.csv')

format = json
   load[gx    ] pd.read_json('/home/acraf/psr/Fdatavalidation-1/PositionPaper/data/Patient_Summary.json')
   load[duckdb] read_json_auto('/home/acraf/psr/Fdatavalidation-1/PositionPaper/data/Patient_Summary.json')
   load[dbt   ] # dbt source (dbt-duckdb): external_location: read_json_auto('/home/acraf/psr/Fdatavalidation-1/PositionPaper/data/Patient_Summary.json')

format = parquet
   load[gx    ] pd.read_parquet('/home/acraf/psr/Fdatavalidation-1/PositionPaper/data/Patient_Summary.parquet')
   load[duckdb] read_parquet('/home/acraf/psr/Fdatavalidation-1/PositionPaper/data/Patient_Summary.parquet')
   

## 4. Run the generated code on CSV, JSON and Parquet

GX expectations run live; the generated dbt tests run as SQL via DuckDB; a DuckDB measure-and-compare gives the reference value and verdict. Same plan, three formats, identical verdicts.

In [7]:
import duckdb

def run_format(plan):
    code = C.generate(plan)
    gx   = C.run_gx(code["gx"], plan)          # generated GX, live
    ref  = C.execute(plan)                      # reference measure-and-compare (format-aware)
    con  = duckdb.connect()
    con.execute(f"CREATE TABLE data AS SELECT * FROM {C.load_code(plan, 'duckdb')}")
    con.execute("CREATE TABLE codes(code VARCHAR)")
    std = C.ref_standard(plan, EX["c-compliance"])
    con.executemany("INSERT INTO codes VALUES (?)", [(x,) for x in C.resolve_codes(std)])
    dbt = {}
    for cid, sql in code["dbt"].items():
        body = "\n".join(l for l in sql.splitlines() if not l.startswith("--"))
        body = body.replace("{{ source('ehds','patient_summary') }}", "data")
        dbt[cid] = not con.execute(body).fetchall()
    return ref, gx, dbt

for label, plan in [("CSV", plan_csv), ("JSON", plan_json), ("PARQUET", plan_parquet)]:
    ref, gx, dbt = run_format(plan)
    print(f"--- {label} ({C.dataset_of(plan)[0]}) ---")
    print(f"  {'check':16s} {'measured':>10s}  {'verdict':8s} {'GX':6s} {'dbt':6s}")
    for cid in sorted(ref):
        v, ok = ref[cid]
        print(f"  {cid:16s} {('—' if v is None else f'{v:10.4f}')}  "
              f"{('SAT' if ok else 'VIOL'):8s} {('PASS' if gx[cid] else 'FAIL'):6s} "
              f"{('PASS' if dbt[cid] else 'FAIL'):6s}")
    print()

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

--- CSV (csv) ---
  check              measured  verdict  GX     dbt   
  c-completeness      98.0000  VIOL     FAIL   FAIL  
  c-compliance        98.1964  SAT      PASS   PASS  
  c-consistency      100.0000  SAT      PASS   PASS  



Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

--- JSON (json) ---
  check              measured  verdict  GX     dbt   
  c-completeness      98.0000  VIOL     FAIL   FAIL  
  c-compliance        98.1964  SAT      PASS   PASS  
  c-consistency      100.0000  SAT      PASS   PASS  

--- PARQUET (parquet) ---
  check              measured  verdict  GX     dbt   
  c-completeness      98.0000  VIOL     FAIL   FAIL  
  c-compliance        98.1964  SAT      PASS   PASS  
  c-consistency      100.0000  SAT      PASS   PASS  



## 5. Provenance of results (DQV)

Each measured value is recorded as a `dqv:QualityMeasurement` on the workflow's `MeasureValue`.

In [8]:
meas = C.measurements_graph(plan_csv, C.execute(plan_csv))
meas.serialize(destination=f"{PIPE}/artifacts/measurements.ttl", format="turtle")
print(meas.serialize(format="turtle"))

@prefix dqv: <http://www.w3.org/ns/dqv#> .
@prefix pc: <https://feed-upc.github.io/DQVP-position/pc#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://example.org/ehds/c-completeness/mval> a dqv:QualityMeasurement ;
    dqv:isMeasurementOf <https://feed-upc.github.io/DQVP-position/dqvp.ttl#NonNullRate> ;
    dqv:value 98.0 ;
    pc:verdict false .

<https://example.org/ehds/c-compliance/mval> a dqv:QualityMeasurement ;
    dqv:isMeasurementOf <https://feed-upc.github.io/DQVP-position/dqvp.ttl#ReferenceConformance> ;
    dqv:value 98.1964 ;
    pc:verdict true .

<https://example.org/ehds/c-consistency/mval> a dqv:QualityMeasurement ;
    dqv:isMeasurementOf <https://feed-upc.github.io/DQVP-position/dqvp.ttl#NullRate> ;
    dqv:value 100.0 ;
    pc:verdict true .




## Summary

A DQR catalogue maps to one DQVP policy, which compiles to one technology-agnostic Policy Checker, then to code for four engines, with the Load code selected per data format. GX and dbt run that code on CSV, JSON and Parquet with identical verdicts, recorded back in DQV. Adding a requirement is one JSON; a format is one load-template row; an engine is check plus load-template rows; a metric shape is one rule.